# Tesla Model 3 Performance — lap time telemetry

This notebook reads Tesla Track Mode telemetry from a configurable directory. Use the CSV dropdown to change sessions and the lap dropdown to inspect any recorded timed lap.

The synchronized explorer places speed, driver inputs, vehicle dynamics, power, and steering graphs above the GPS track. Drag the yellow position handle on the track (or click the trace / drag the time slider) to move every graph to the corresponding recorded sample.

## Environment

The draggable map uses the Jupyter Matplotlib widget backend. From the repository root, install the dependencies once with <code>python -m pip install -r requirements.txt</code>, then restart the notebook kernel.

In [ ]:
from importlib.util import find_spec
from pathlib import Path

if find_spec("ipympl") is None:
    raise ModuleNotFoundError(
        "The draggable track view needs ipympl. Run %pip install -r requirements.txt, "
        "restart the kernel, and then run the notebook again."
    )

get_ipython().run_line_magic("matplotlib", "widget")

from IPython.display import display

from telemetry_utils import (
    TelemetryDashboard,
    build_sector_summary,
    format_lap_time,
)


: 

## Configuration

Change <code>DATA_DIR</code> to point at another folder. The refresh button rescans CSV files directly inside that folder. Leave <code>DEFAULT_FILE</code> as <code>None</code> to open the newest filename, or set it to a CSV filename.

In [ ]:
DATA_DIR = Path("data/20260714")
N_SECTORS = 3
DEFAULT_FILE = None  # Example: "telemetry-v1-2026-07-14-18_44_10.csv"


## Interactive lap explorer

1. Choose a telemetry CSV. Files that contain only Lap 0 remain selectable and show a clear empty state.
2. Choose a timed lap and preferred speed unit.
3. Drag the yellow handle along the GPS trace. It snaps to the nearest Tesla sample and updates the time cursor, graph markers, slider, and exact telemetry readout.

The session summary, fastest lap, equal-distance sector estimate, lap table, and plots all refresh when the selected CSV changes.

In [ ]:
dashboard = TelemetryDashboard(
    DATA_DIR,
    n_sectors=N_SECTORS,
    default_file=DEFAULT_FILE,
)
dashboard.display()


## Optional programmatic access

The dashboard keeps its currently loaded frames available for further analysis. After changing a dropdown, rerun the next cell if you want fresh local variable names.

In [ ]:
selected_session = dashboard.telemetry
timed = dashboard.timed
lap_summary = dashboard.lap_summary
selected_lap_data = dashboard.lap_data

display(lap_summary)

if not timed.empty:
    sector_summary, best_sectors = build_sector_summary(timed, N_SECTORS)
    theoretical_ms = best_sectors["Sector Time (ms)"].sum()
    print(f"Theoretical {N_SECTORS}-sector lap: {format_lap_time(theoretical_ms)}")
    display(sector_summary)
    display(best_sectors)


## Interpretation notes

- Lap 0 is session/out-lap data in these recordings because its elapsed time remains zero, so it is excluded from timed-lap controls.
- A lap time is the maximum recorded elapsed time for that positive lap ID. The notebook does not guess whether a long final lap was interrupted or became an in-lap.
- Sector boundaries are equal fractions of measured GPS distance; GPS noise and racing-line differences can shift equivalent boundaries slightly between laps.
- Negative brake sensor offsets are preserved in the loaded data and clipped to zero only in the brake plot/readout.
- The track is drawn directly from GPS coordinates without a basemap, keeping the explorer offline and reproducible.